In [ ]:
import pandas as pd
import mlflow
import mlflow.sklearn

df = pd.read_csv("processedData.csv", parse_dates=["purchase_date"])

mlflow.set_experiment("Ecommerce_Demand_Forecasting")

In [ ]:
#######Demand Feature Engineering Module####### - Eades

## Convert purchase date to datetime and create monthly time period
df["purchase_date"] = pd.to_datetime(df["purchase_date"])
df["month"] = df["purchase_date"].dt.to_period("M").dt.to_timestamp()

## Aggregate transaction records to product/month level to create product demand
## Demand is defined as the number of purchases for each product in each month
df_demand = (
    df.groupby(["product_id", "month"])
      .agg(
          demand=("user_id", "count"),
          price=("price", "mean"),
          discount=("discount", "mean"),
          final_price=("final_price", "mean"),
          rating=("rating", "mean"),
          review_count=("review_count", "mean"),
          stock=("stock", "mean"),
          seller_rating=("seller_rating", "mean"),
          shipping_time_days=("shipping_time_days", "mean"),
          category=("category", "first"),
          subcategory=("subcategory", "first"),
          brand=("brand", "first"))
      .reset_index())

In [ ]:
## Sort by product and date before creating historical demand features
df_demand = df_demand.sort_values(["product_id", "month"])

## Historical purchase activity features used to predict future product demand
df_demand["demand_last_month"] = df_demand.groupby("product_id")["demand"].shift(1)
df_demand["demand_2_months_ago"] = df_demand.groupby("product_id")["demand"].shift(2)
df_demand["avg_demand_last_3_months"] = (
    df_demand.groupby("product_id")["demand"]
             .transform(lambda x: x.shift(1).rolling(3).mean()))

## Month number allows the models to account for monthly differences
df_demand["month_number"] = df_demand["month"].dt.month

## Remove records that do not yet have enough historical data for lag features
df_model = df_demand.dropna().copy()

print(df_model.head())
print("Model records:", len(df_model))

In [ ]:
#######Model Feature Selection and Preprocessing####### - Eades

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

## Numeric features identified in proposal plus historical purchase activity
numeric_features = ["price", "discount", "final_price", "rating", "review_count",
    "stock", "seller_rating", "shipping_time_days",
    "demand_last_month", "demand_2_months_ago",
    "avg_demand_last_3_months", "month_number"]

## Product characteristics that require categorical encoding
categorical_features = ["category", "subcategory", "brand"]

## Preprocessing kept in pipeline to maintain reproducibility
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)])

In [ ]:
#######Training Validation Testing####### - Eades

## Sort chronologically because the project predicts future product demand
df_model = df_model.sort_values("month").reset_index(drop=True)

## Split by unique months so a month is not divided between data sets
model_months = sorted(df_model["month"].unique())
training_end = int(len(model_months) * 0.80)
validation_end = int(len(model_months) * 0.90)

training_months = model_months[:training_end]
validation_months = model_months[training_end:validation_end]
testing_months = model_months[validation_end:]

df_model_training = df_model[df_model["month"].isin(training_months)].copy()
df_model_validation = df_model[df_model["month"].isin(validation_months)].copy()
df_model_testing = df_model[df_model["month"].isin(testing_months)].copy()

X_training = df_model_training[numeric_features + categorical_features]
y_training = df_model_training["demand"]

X_validation = df_model_validation[numeric_features + categorical_features]
y_validation = df_model_validation["demand"]

X_testing = df_model_testing[numeric_features + categorical_features]
y_testing = df_model_testing["demand"]

print("Training records:", len(df_model_training))
print("Validation records:", len(df_model_validation))
print("Testing records:", len(df_model_testing))
print("Training date range:", df_model_training["month"].min(), "to", df_model_training["month"].max())
print("Validation date range:", df_model_validation["month"].min(), "to", df_model_validation["month"].max())
print("Testing date range:", df_model_testing["month"].min(), "to", df_model_testing["month"].max())

In [ ]:
#######Linear Regression Model####### - Eades

from sklearn.linear_model import LinearRegression

## Linear Regression baseline model
LinearRegressionModel = Pipeline([("preprocessor", preprocessor),
    ("model", LinearRegression())])

LinearRegressionModel.fit(X_training, y_training)
LinearRegressionValidationPredictions = LinearRegressionModel.predict(X_validation)

In [ ]:
#######Random Forest Regression Model####### - Eades

from sklearn.ensemble import RandomForestRegressor

## Random Forest model to capture non-linear relationships
RandomForestModel = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=25,
        n_jobs=-1))])

RandomForestModel.fit(X_training, y_training)
RandomForestValidationPredictions = RandomForestModel.predict(X_validation)

In [ ]:
#######XGBoost Regression Model####### - Eades

from xgboost import XGBRegressor

## XGBoost model included as an additional non-linear model for comparison
XGBoostModel = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=25,
        n_jobs=-1))])

XGBoostModel.fit(X_training, y_training)
XGBoostValidationPredictions = XGBoostModel.predict(X_validation)

In [ ]:
#######Model Performance Evaluation####### - Eades

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from IPython.display import display

## Function calculates the three technical metrics identified in the proposal
def EvaluateModel(y_actual, y_predicted):
    return {"MAE": mean_absolute_error(y_actual, y_predicted),
        "RMSE": np.sqrt(mean_squared_error(y_actual, y_predicted)),
        "R2": r2_score(y_actual, y_predicted)}

LinearRegressionResults = EvaluateModel(y_validation, LinearRegressionValidationPredictions)
RandomForestResults = EvaluateModel(y_validation, RandomForestValidationPredictions)
XGBoostResults = EvaluateModel(y_validation, XGBoostValidationPredictions)

#####MLFlow Tracking#####

#Liner Regression
with mlflow.start_run(run_name="Linear_Regression"):
    mlflow.log_metric("validation_MAE",
        LinearRegressionResults["MAE"])
    mlflow.log_metric("validation_RMSE",
        LinearRegressionResults["RMSE"])
    mlflow.log_metric("validation_R2",
        LinearRegressionResults["R2"])
    mlflow.sklearn.log_model(LinearRegressionModel,
        name="linear_regression_model")

# Random Forest
with mlflow.start_run(run_name="Random_Forest"):
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("random_state", 25)
    mlflow.log_metric("validation_MAE",
        RandomForestResults["MAE"])
    mlflow.log_metric("validation_RMSE",
        RandomForestResults["RMSE"])
    mlflow.log_metric("validation_R2",
        RandomForestResults["R2"])
    mlflow.sklearn.log_model(
        RandomForestModel,
        name="random_forest_model")


# XGBoost
with mlflow.start_run(run_name="XGBoost"):
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("random_state", 25)
    mlflow.log_metric("validation_MAE",
        XGBoostResults["MAE"])
    mlflow.log_metric("validation_RMSE",
        XGBoostResults["RMSE"])
    mlflow.log_metric("validation_R2",
        XGBoostResults["R2"])
    mlflow.sklearn.log_model(XGBoostModel,
        name="xgboost_model")

ModelComparison = pd.DataFrame(
    [LinearRegressionResults, RandomForestResults, XGBoostResults],
    index=["Linear Regression", "Random Forest", "XGBoost"])

## Rounded display table while retaining full precision for model selection
ModelComparisonDisplay = ModelComparison.copy()
ModelComparisonDisplay["MAE"] = ModelComparisonDisplay["MAE"].round(2)
ModelComparisonDisplay["RMSE"] = ModelComparisonDisplay["RMSE"].round(2)
ModelComparisonDisplay["R2"] = ModelComparisonDisplay["R2"].round(3)
ModelComparisonDisplay.index.name = "Model"

display(ModelComparisonDisplay.style
        .set_caption("Validation Performance by Model")
        .format({"MAE": "{:.2f}", "RMSE": "{:.2f}", "R2": "{:.3f}"})
        .set_properties(**{"text-align": "center"}))

In [ ]:
#######Final Model Selection and Testing####### - Eades

## Select the model with the lowest validation RMSE
BestModelName = ModelComparison["RMSE"].idxmin()

ModelDictionary = {"Linear Regression": LinearRegressionModel,
    "Random Forest": RandomForestModel,
    "XGBoost": XGBoostModel}

BestModel = ModelDictionary[BestModelName]

## Test set used only after model comparison/selection is complete
FinalTestPredictions = BestModel.predict(X_testing)
FinalTestResults = EvaluateModel(y_testing, FinalTestPredictions)

FinalTestResultsTable = pd.DataFrame(
    [FinalTestResults],
    index=[BestModelName])
FinalTestResultsTable.index.name = "Selected Model"

#####MLFlow Final Test#####
with mlflow.start_run(run_name="Final_Selected_Model"):

    mlflow.log_param("selected_model", BestModelName)
    mlflow.log_metric("test_MAE", FinalTestResults["MAE"])
    mlflow.log_metric("test_RMSE",FinalTestResults["RMSE"])
    mlflow.log_metric("test_R2",FinalTestResults["R2"])
    mlflow.sklearn.log_model(BestModel, name="final_selected_model")


display(
    FinalTestResultsTable.style
        .set_caption("Final Test Performance")
        .format({"MAE": "{:.2f}", "RMSE": "{:.2f}", "R2": "{:.3f}"})
        .set_properties(**{"text-align": "center"}))

In [ ]:
#######Model Visualization####### - Eades

import matplotlib.pyplot as plt

## Colors used for the three models
model_colors = ["#1F77B4", "#D62728", "#2CA02C"]

## Validation RMSE comparison
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(ModelComparison.index,
    ModelComparison["RMSE"],
    color=model_colors,
    edgecolor="black",
    linewidth=0.8)

ax.set_ylabel("RMSE", fontsize=12)
ax.set_title("Validation RMSE by Model", fontsize=14, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", labelsize=11)

for bar, value in zip(bars, ModelComparison["RMSE"]):
    ax.text(bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=10)

plt.tight_layout()
plt.show()

## Actual versus predicted demand for the selected model
fig, ax = plt.subplots(figsize=(6.5, 6))

ax.scatter(
    y_testing,
    FinalTestPredictions,
    color="#1F77B4",
    edgecolor="black",
    linewidth=0.3,
    alpha=0.65,
    s=32)

minimum_value = min(y_testing.min(), FinalTestPredictions.min())
maximum_value = max(y_testing.max(), FinalTestPredictions.max())

ax.plot([minimum_value, maximum_value],
    [minimum_value, maximum_value],
    color="#D62728",
    linewidth=1.8,
    linestyle="--")

ax.set_xlabel("Actual Product Demand", fontsize=12)
ax.set_ylabel("Predicted Product Demand", fontsize=12)
ax.set_title(
    BestModelName + " - Actual vs Predicted Product Demand",
    fontsize=14,
    fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", labelsize=11)

plt.tight_layout()
plt.show()

In [ ]:
import joblib
import json

joblib.dump(BestModel, "best_model.joblib")

with open("model_features.json", "w") as file:
    json.dump(numeric_features + categorical_features, file)

X_testing.head(20).to_csv("sample_prediction_inputs.csv", index=False)